# IDOM ハンズオン Day 1

## 内容
1. **環境セットアップ** — データベース・テーブル・サンプルデータの作成
2. **Dynamic Table** — 宣言的データパイプラインの構築
3. **Cortex AI Functions** — SQL だけで AI 分析（CLASSIFY_TEXT / COMPLETE）
4. **Marketplace** — 外部データ（天気）の取得と活用

---
## 0. 環境セットアップ

ハンズオンの前提環境を構築します。

**`00_setup.sql` を Snowsight のワークシートで実行してください。**
（データ量が多いため、ワークシートでの実行を推奨します）

作成されるオブジェクト:
- `IDOM_HANDSON` データベース
- `RAW_DATA` スキーマ（STORES / CONTRACTS / INSURANCE / ACTIVITY_LOGS）
- `DATA_MART` スキーマ（空 → ハンズオンで構築）
- `AGENTS` スキーマ（空 → Day 2 で構築）

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
%%sql -r dataframe_2
-- セットアップ済みか確認
SELECT 'STORES' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM IDOM_HANDSON.RAW_DATA.STORES
UNION ALL
SELECT 'CONTRACTS', COUNT(*) FROM IDOM_HANDSON.RAW_DATA.CONTRACTS
UNION ALL
SELECT 'INSURANCE', COUNT(*) FROM IDOM_HANDSON.RAW_DATA.INSURANCE
UNION ALL
SELECT 'ACTIVITY_LOGS', COUNT(*) FROM IDOM_HANDSON.RAW_DATA.ACTIVITY_LOGS;

---
## 1. Dynamic Table — 売上実績パイプラインを構築

Dynamic Table は**宣言的**にデータパイプラインを定義する機能です。
ソーステーブルが更新されると、自動的にリフレッシュされます。

ポイント:
- `TARGET_LAG`: データの鮮度（好きな時間をご入力）

### Step 1: Dynamic Table を作成

CONTRACTS × STORES × INSURANCE を結合し、店舗別・月別の売上KPIを集計します。

`問題：TARGET_LAGを皆さんで好きに設定してください`

In [ ]:
USE SCHEMA IDOM_HANDSON.DATA_MART;

CREATE OR REPLACE DYNAMIC TABLE IDOM_HANDSON.DATA_MART.DT_SALES_PERFORMANCE
    TARGET_LAG = '1minute' --問題：皆さんで好きに設定してください
    WAREHOUSE = COMPUTE_WH
AS
SELECT
    s.STORE_ID,
    s.STORE_NAME,
    s.REGION,
    s.PREFECTURE,
    s.STORE_TYPE,
    s.TARGET_MONTHLY_SALES,
    s.TARGET_MONTHLY_CONTRACTS,
    DATE_TRUNC('month', c.CONTRACT_DATE) AS CONTRACT_MONTH,
    COUNT(DISTINCT c.CONTRACT_ID) AS CONTRACT_COUNT,
    SUM(c.CONTRACT_PRICE) AS TOTAL_SALES,
    AVG(c.CONTRACT_PRICE) AS AVG_CONTRACT_PRICE,
    SUM(c.TAX_AMOUNT) AS TOTAL_TAX,
    SUM(c.LOAN_AMOUNT) AS TOTAL_LOAN_AMOUNT,
    COUNT(CASE WHEN c.CONTRACT_STATUS = '納車済' THEN 1 END) AS DELIVERED_COUNT,
    COUNT(CASE WHEN c.CONTRACT_STATUS = '手続中' THEN 1 END) AS PROCESSING_COUNT,
    COUNT(CASE WHEN c.CONTRACT_STATUS = 'キャンセル' THEN 1 END) AS CANCELLED_COUNT,
    COUNT(DISTINCT i.INSURANCE_ID) AS INSURANCE_COUNT,
    SUM(i.PREMIUM_AMOUNT) AS TOTAL_INSURANCE_PREMIUM,
    ROUND(COUNT(DISTINCT i.INSURANCE_ID) * 100.0 / NULLIF(COUNT(DISTINCT c.CONTRACT_ID), 0), 1) AS INSURANCE_ATTACH_RATE
FROM IDOM_HANDSON.RAW_DATA.CONTRACTS c
JOIN IDOM_HANDSON.RAW_DATA.STORES s ON c.STORE_ID = s.STORE_ID
LEFT JOIN IDOM_HANDSON.RAW_DATA.INSURANCE i ON c.CONTRACT_ID = i.CONTRACT_ID
WHERE c.CONTRACT_DATE IS NOT NULL
GROUP BY s.STORE_ID, s.STORE_NAME, s.REGION, s.PREFECTURE, s.STORE_TYPE,
         s.TARGET_MONTHLY_SALES, s.TARGET_MONTHLY_CONTRACTS,
         DATE_TRUNC('month', c.CONTRACT_DATE);

In [ ]:
%%sql -r dataframe_4
SELECT *
FROM IDOM_HANDSON.DATA_MART.DT_SALES_PERFORMANCE
ORDER BY CONTRACT_MONTH DESC, TOTAL_SALES DESC;

### Step 2: Dynamic Table の状態確認

リフレッシュの状態や履歴を確認します。

In [ ]:
%%sql -r dataframe_5
SHOW DYNAMIC TABLES IN SCHEMA IDOM_HANDSON.DATA_MART;

### Step 3: ソースデータを更新して自動リフレッシュを体験

新しい契約データを追加 → 手動リフレッシュ → 反映を確認します。

In [ ]:
%%sql -r dataframe_6
-- 更新前の東京本店の件数を確認
SELECT STORE_NAME, CONTRACT_MONTH, CONTRACT_COUNT
FROM IDOM_HANDSON.DATA_MART.DT_SALES_PERFORMANCE
WHERE STORE_NAME = 'ガリバー東京本店'
ORDER BY CONTRACT_MONTH DESC LIMIT 3;

In [ ]:
%%sql -r dataframe_7
INSERT INTO IDOM_HANDSON.RAW_DATA.CONTRACTS
(CONTRACT_ID, NEGOTIATION_ID, STORE_ID, CUSTOMER_NAME, CAR_NAME,
 CONTRACT_TYPE, CONTRACT_PRICE, TAX_AMOUNT, PAYMENT_METHOD,
 LOAN_AMOUNT, LOAN_TERM_MONTHS, CONTRACT_DATE, DELIVERY_DATE, CONTRACT_STATUS)
VALUES
('D9999', 'N9999', 'S001', 'テスト太郎', 'トヨタ プリウス',
 '現金一括', 3000000, 300000, '銀行振込',
 0, 0, '2026-04-08', '2026-04-21', '手続中');

In [ ]:
%%sql -r dataframe_8
--設定したTarget LAGが待てない方はこちらを実行
ALTER DYNAMIC TABLE IDOM_HANDSON.DATA_MART.DT_SALES_PERFORMANCE REFRESH;

In [ ]:
%%sql -r dataframe_9
-- 更新後の件数を確認（リフレッシュ完了後に実行）
SELECT STORE_NAME, CONTRACT_MONTH, CONTRACT_COUNT
FROM IDOM_HANDSON.DATA_MART.DT_SALES_PERFORMANCE
WHERE STORE_NAME = 'ガリバー東京本店'
ORDER BY CONTRACT_MONTH DESC LIMIT 3;

---
## 2. Cortex AI Functions — SQL だけで AI 分析

Snowflake Cortex の AI 関数を使って、商談活動履歴をSQL だけで AI 分析します。
ACTIVITY_LOGS は**半構造化データ（VARIANT型・JSON）**で格納されています。

体験する内容:
- **Step 0**: 半構造化データの確認と展開
- **Step 1**: CLASSIFY_TEXT — テキスト分類
- **Step 2**: COMPLETE — AI 営業コーチ

### Step 0: 半構造化データの確認と展開

ACTIVITY_LOGS テーブルは VARIANT 型の `RAW_DATA` カラム1列に JSON が格納されています。
VARIANT 型からは **`:`** でキーを指定、**`::TYPE`** でキャストしてアクセスできます。

In [ ]:
%%sql -r dataframe_10
USE SCHEMA IDOM_HANDSON.RAW_DATA;

SELECT RAW_DATA
FROM ACTIVITY_LOGS
LIMIT 3;

In [ ]:
%%sql -r dataframe_11
SELECT
    RAW_DATA:activity_id::VARCHAR AS ACTIVITY_ID,
    RAW_DATA:customer_name::VARCHAR AS CUSTOMER_NAME,
    RAW_DATA:activity_type::VARCHAR AS ACTIVITY_TYPE,
    RAW_DATA:subject::VARCHAR AS SUBJECT,
    LEFT(RAW_DATA:body::VARCHAR, 100) || '...' AS BODY_PREVIEW
FROM ACTIVITY_LOGS
ORDER BY ACTIVITY_ID
LIMIT 10;

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE VIEW IDOM_HANDSON.RAW_DATA.V_ACTIVITY_LOGS AS
SELECT
    RAW_DATA:activity_id::VARCHAR AS ACTIVITY_ID,
    RAW_DATA:negotiation_id::VARCHAR AS NEGOTIATION_ID,
    RAW_DATA:store_id::VARCHAR AS STORE_ID,
    RAW_DATA:customer_name::VARCHAR AS CUSTOMER_NAME,
    RAW_DATA:assigned_user_name::VARCHAR AS ASSIGNED_USER_NAME,
    RAW_DATA:activity_type::VARCHAR AS ACTIVITY_TYPE,
    RAW_DATA:activity_date::TIMESTAMP_NTZ AS ACTIVITY_DATE,
    RAW_DATA:subject::VARCHAR AS SUBJECT,
    RAW_DATA:body::VARCHAR AS BODY,
    RAW_DATA:created_at::TIMESTAMP_NTZ AS CREATED_AT
FROM ACTIVITY_LOGS;

In [ ]:
%%sql -r dataframe_13
SELECT * FROM V_ACTIVITY_LOGS ORDER BY ACTIVITY_ID LIMIT 10;

### Step 1: CLASSIFY_TEXT — 商談フェーズの自動分類

活動内容テキストを事前定義のカテゴリに自動分類します。
モデルの学習なしで、SQL だけで分類できるのがポイントです。

`問題2：分類したいカテゴリを入力してください`

In [ ]:
SELECT
    ACTIVITY_ID,
    CUSTOMER_NAME,
    SUBJECT,
    BODY,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        BODY,
        ['XXX', 'AAA'] --問題：分類したいカテゴリを入力してください
    ):label::VARCHAR AS CLASSIFIED_PHASE
FROM V_ACTIVITY_LOGS
ORDER BY ACTIVITY_ID
LIMIT 30;

In [ ]:
SELECT
    ACTIVITY_ID,
    CUSTOMER_NAME,
    SUBJECT,
    BODY,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        BODY,
        ['初回問合せ', '商談進行中', '契約完了', '失注・見送り', '価格交渉'] --回答
    ):label::VARCHAR AS CLASSIFIED_PHASE
FROM V_ACTIVITY_LOGS
ORDER BY ACTIVITY_ID
LIMIT 30;

### Step 2: COMPLETE — AI 営業コーチ

LLM が商談活動を読み取り、担当営業へ **「次にやるべきアクション」** を具体的にアドバイスします。
まるで AI コーチが隣にいるような体験！

In [ ]:
%%sql -r dataframe_16
SELECT
    ACTIVITY_ID,
    CUSTOMER_NAME,
    SUBJECT,
    body,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-70b',
        'あなたは中古車販売のトップ営業コンサルタントです。' ||
        '以下の商談活動記録を読んで、担当営業マンへ次にやるべき具体的なアクションを' ||
        '短く箇条書きのみでアドバイスしてください。' ||
         CHR(10) || CHR(10) ||
        '【件名】' || SUBJECT || CHR(10) ||
        '【内容】' || BODY
    ) AS AI_COACHING
FROM V_ACTIVITY_LOGS
WHERE ACTIVITY_ID IN ('ACT0001', 'ACT0006', 'ACT0011', 'ACT0016', 'ACT0020')
ORDER BY ACTIVITY_ID;

---
## 3. Marketplace — 外部データの取得

Snowflake Marketplace から外部データセット（天気データ）を取得し、
売上実績 Dynamic Table と組み合わせる方法を学びます。

### 手順（Snowsight UI で操作）

1. 左メニュー → **Data Products** → **Marketplace**
2. 検索バーに **「Weather Source」** と入力
3. **「Weather Source LLC: frostbyte」** を選択（無料 Standard 版）
4. **Get** → データベース名はデフォルトで **Get** → 完了

> Marketplace データは**コピーではなくライブ共有**のため、提供元が更新すると自動で最新データが反映されます。

In [ ]:
%%sql -r dataframe_17
SHOW SCHEMAS IN DATABASE WEATHER_SOURCE_LLC_FROSTBYTE;

In [ ]:
%%sql -r dataframe_18
SELECT *
FROM WEATHER_SOURCE_LLC_FROSTBYTE.ONPOINT_ID.HISTORY_DAY
WHERE COUNTRY = 'JP'
ORDER BY DATE_VALID_STD
limit 100;

---
## Day 1 まとめ

| 項目 | 学んだこと |
|------|----------|
| Dynamic Table | 宣言的パイプライン定義、自動リフレッシュ |
| VARIANT / JSON | 半構造化データの格納と展開（`:` アクセサ, `::TYPE` キャスト） |
| CLASSIFY_TEXT | テキストを事前定義カテゴリに自動分類 |
| COMPLETE | LLM を使ったテキスト生成（営業コーチング） |
| Marketplace | 外部データのライブ共有取得 |

**Day 2** では Cortex Search Service と Cortex Agent を構築し、
構造化データ × 非構造化データを統合した AI アシスタントを完成させます。